In [1]:
# ─── ЯЧЕЙКА 1: Проверка окружения ───────────────────────────────────────────
import torch
import transformers
import accelerate
import bitsandbytes as bnb

print("torch:         ", torch.__version__)
print("transformers:  ", transformers.__version__)
print("accelerate:    ", accelerate.__version__)
print("bitsandbytes:  ", bnb.__version__)

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:           ", torch.cuda.get_device_name(0))
    print("VRAM total:    ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    print("VRAM free:     ", round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")


D:\bogdanov\PyProjects\Agent_system1\.venv_py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch:          2.7.1+cu118
transformers:   4.52.4
accelerate:     1.13.0
bitsandbytes:   0.49.2

CUDA available: True
GPU:            NVIDIA GeForce RTX 3060
VRAM total:     12.0 GB
VRAM free:      10.98 GB


In [2]:
# ─── ЯЧЕЙКА 2: Загрузка токенайзера ─────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_PATH = r"D:\bogdanov\PyProjects\Agents_project\Models\Qwen2.5-14B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

print("Tokenizer loaded")
print("Vocab size:    ", tokenizer.vocab_size)
print("Chat template: ", "YES" if tokenizer.chat_template else "NO")


Tokenizer loaded
Vocab size:     151643
Chat template:  YES


In [3]:
# ─── ЯЧЕЙКА 3: Загрузка модели (4-bit) ──────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True
)

model.eval()

print("Model loaded on:", next(model.parameters()).device)
print("Model type:     ", type(model).__name__)
print("VRAM used:      ", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


Loading checkpoint shards: 100%|██████████| 8/8 [04:09<00:00, 31.23s/it]

Model loaded on: cuda:0
Model type:      Qwen2ForCausalLM
VRAM used:       9.31 GB


In [4]:
# ─── ЯЧЕЙКА 4: Конфиг + registry.json + BM25 ────────────────────────────────
import json
import sqlite3
import re
from rank_bm25 import BM25Okapi

REGISTRY_PATH = r"D:\bogdanov\PyProjects\Agents_project\registry.json"
DB_PATH       = r"D:\bogdanov\PyProjects\Agents_project\commands.db"


def load_registry(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_connection():
    return sqlite3.connect(DB_PATH)


def load_all_commands():
    with get_connection() as conn:
        rows = conn.execute(
            "SELECT name, command, functionality, example, usage_example FROM commands"
        ).fetchall()
    result = []
    for r in rows:
        result.append({
            "name":          r[0],
            "command":       json.loads(r[1]),
            "functionality": r[2],
            "example":       r[3],
            "usage_example": json.loads(r[4]),
        })
    return result


_bm25_index  = None
_bm25_corpus = []


def tokenize(text):
    return re.findall(r"\w+", text.lower())


def build_bm25_index():
    global _bm25_index, _bm25_corpus
    _bm25_corpus = load_all_commands()
    tokenized = [
        tokenize(c["functionality"] + " " + c["example"])
        for c in _bm25_corpus
    ]
    _bm25_index = BM25Okapi(tokenized)
    print("BM25 index built:", len(_bm25_corpus), "commands")


def retrieve_commands(user_request, top_k=3):
    global _bm25_index
    if _bm25_index is None:
        build_bm25_index()
    tokens  = tokenize(user_request)
    scores  = _bm25_index.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [dict(_bm25_corpus[i], score=round(float(scores[i]), 4)) for i in top_idx]


registry = load_registry(REGISTRY_PATH)
build_bm25_index()

print("Загружено агентов:", len(registry["agents"]))
for a in registry["agents"]:
    print("  →", a["name"])
print("Registry + BM25 ready")


BM25 index built: 7 commands
Загружено агентов: 5
  → FinanceAgent
  → ResearchAgent
  → PlanningAgent
  → DataAgent
  → SummaryAgent
Registry + BM25 ready


In [18]:
# ─── ЯЧЕЙКА 5: Системный промпт Handler-а ───────────────────────────────────
import json


def build_handler_system_prompt(user_request, registry):

    agents_lines = []
    for a in registry["agents"]:
        agents_lines.append("  - " + a["name"] + ": " + a["system_prompt"])
    agents_block = "\n".join(agents_lines)

    top_commands = retrieve_commands(user_request, top_k=3)

    cmd_lines = []
    for c in top_commands:
        cmd_lines.append("  name:          " + c["name"])
        cmd_lines.append("  functionality: " + c["functionality"])
        cmd_lines.append("  example:       " + c["example"])
        cmd_lines.append("  template:      " + json.dumps(c["command"],       ensure_ascii=False))
        cmd_lines.append("  usage_example: " + json.dumps(c["usage_example"], ensure_ascii=False))
        cmd_lines.append("")
    commands_block = "\n".join(cmd_lines)

    prompt = (
    "You are the central Handler of a distributed multi-agent system.\n"
    "\n"
    "You are NOT a general chatbot.\n"
    "You operate ONLY within the scope of this multi-agent system.\n"
    "\n"
    "You have TWO allowed response modes:\n"
    "\n"
    "MODE 1 — COMMAND MODE:\n"
    "When the user asks to do something with agents or data — decompose the request\n"
    "into one or more engine commands and output a JSON array.\n"
    "\n"
    "MODE 2 — SYSTEM Q&A MODE:\n"
    "Triggers: any question about agents, commands, system capabilities, how it works.\n"
    "Examples of MODE 2 questions:\n"
    "  - 'what agents are in the system'\n"
    "  - 'what does FinanceAgent do'\n"
    "  - 'list all commands'\n"
    "  - 'what can this system do'\n"
    "  - 'give me the functionality of each agent'\n"
    "\n"
    "In MODE 2 you MUST:\n"
    "  - Answer DIRECTLY in natural language\n"
    "  - Use ONLY information from AVAILABLE AGENTS and AVAILABLE COMMANDS in this prompt\n"
    "  - NEVER generate any JSON\n"
    "  - NEVER call extract, add, or any other command\n"
    "  - End with one suggested follow-up action\n"
    "\n"
    "MODE 2 response format:\n"
    "<SYSTEM INFO>\n"
    "your answer here\n"
    "</SYSTEM INFO>\n"
    "\n"
    "CRITICAL: if the user asks ABOUT the system — answer from the prompt, do NOT execute commands.\n"
    "\n"
    "AVAILABLE AGENTS (from registry):\n"
    + agents_block +
    "\n\n"
    "AVAILABLE COMMANDS (retrieved for this request):\n"
    + commands_block +
    "\n\n"
    "YOUR TASK:\n"
    "1. Determine which mode applies to the user request\n"
    "2. MODE 1: decompose into commands, fill templates, output JSON array\n"
    "3. MODE 2: answer the system question briefly, suggest a follow-up action\n"
    "4. Commands must follow usage_example format\n"
    "5. Command sequence reflects execution order\n"
    "\n"
    "OUTPUT FORMAT for MODE 1 (strict JSON array):\n"
    "[\n"
    "  { ...command_1... },\n"
    "  { ...command_2... }\n"
    "]\n"
    "\n"
    "RULES:\n"
    "- Agent names must exactly match AVAILABLE AGENTS\n"
    "- Command actions must exactly match AVAILABLE COMMANDS\n"
    "- Prefer minimal number of commands to fulfill the request\n"
    "- If one command is enough output array with one element\n"
    "- For MODE 1 output ONLY the JSON array, no markdown, no explanation\n"
    "- For MODE 2 keep the answer short and always redirect to a system action\n"
    "- NEVER answer questions unrelated to this system\n"
    "- If the request is completely off-topic respond strictly with:\n"
    "  <I only operate within this multi-agent system. Please ask about agents, commands, or data.>\n"
        )
    return prompt


print("build_handler_system_prompt() ready")


build_handler_system_prompt() ready


In [6]:
# ─── ЯЧЕЙКА 6: Handler inference — запрос → JSON-команды ────────────────────
import torch
import json
import re


def handler_generate(user_request, registry):
    system_prompt = build_handler_system_prompt(user_request, registry)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_request},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if match:
        raw = match.group(0)

    try:
        commands = json.loads(raw)
        if not isinstance(commands, list):
            commands = [commands]
    except json.JSONDecodeError:
        print("[WARN] Не удалось распарсить JSON:")
        print(raw)
        commands = []

    return commands, raw


print("handler_generate() ready")


handler_generate() ready


In [7]:
# ─── ЯЧЕЙКА 7: Compiler inference — ответ движка → финальный ответ ──────────
import torch
import json


def compile_answer(user_request, commands, engine_response):
    compile_system = (
        "You are a response compiler for a multi-agent system. "
        "Summarize engine results into a clear natural language answer. "
        "Answer in the same language as the user request."
    )

    compile_user = (
        "USER REQUEST:\n" + user_request + "\n\n"
        "COMMANDS SENT TO ENGINE:\n" + json.dumps(commands, ensure_ascii=False, indent=2) + "\n\n"
        "ENGINE RESPONSE:\n" + json.dumps(engine_response, ensure_ascii=False, indent=2) + "\n\n"
        "Write a concise final answer based strictly on ENGINE RESPONSE data."
    )

    messages = [
        {"role": "system", "content": compile_system},
        {"role": "user",   "content": compile_user},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()


print("compile_answer() ready")


compile_answer() ready


In [19]:
# ─── ЯЧЕЙКА 8: REPL (заглушка движка — ответ вводишь вручную) ───────────────
import json

print("Handler REPL")
print("Введи 'exit' для выхода\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ("exit", "quit"):
        print("Выход.")
        break

    # Шаг 1: Handler → JSON-команды
    print("\n[1/3] Handler анализирует запрос...")
    commands, raw = handler_generate(user_input, registry)

    if not commands:
        print("[ERROR] Не удалось сгенерировать команды\n")
        continue

    print("\n[Handler → Engine]", len(commands), "команд(а):")
    print(json.dumps(commands, ensure_ascii=False, indent=2))

    # Шаг 2: вводишь ответ движка вручную
    print("\n[2/3] Передай команды движку вручную.")
    print("Вставь JSON-ответ движка и нажми Enter дважды:\n")

    lines = []
    while True:
        line = input()
        if line == "":
            break
        lines.append(line)

    engine_raw = "\n".join(lines).strip()

    try:
        engine_response = json.loads(engine_raw)
    except json.JSONDecodeError:
        print("[WARN] Не удалось распарсить ответ движка, используем как строку")
        engine_response = {"raw": engine_raw}

    print("\n[Engine response получен]")

    # Шаг 3: Compiler → финальный ответ
    print("\n[3/3] Компиляция финального ответа...")
    final = compile_answer(user_input, commands, engine_response)

    print("\n" + "=" * 60)
    print("Handler:", final)
    print("=" * 60 + "\n")


Handler REPL
Введи 'exit' для выхода


[1/3] Handler анализирует запрос...
[WARN] Не удалось распарсить JSON:
<SYSTEM INFO>
FinanceAgent анализирует финансы, ResearchAgent проводит исследования, PlanningAgent планирует задачи, DataAgent обрабатывает данные, SummaryAgent создает сводки.
</SYSTEM INFO>
[ERROR] Не удалось сгенерировать команды


[1/3] Handler анализирует запрос...
[WARN] Не удалось распарсить JSON:
<SYSTEM INFO>
FinanceAgent анализирует финансы, ResearchAgent проводит исследования, PlanningAgent планирует задачи, DataAgent обрабатывает данные, SummaryAgent создает сводки.
</SYSTEM INFO>
[ERROR] Не удалось сгенерировать команды


[1/3] Handler анализирует запрос...
[WARN] Не удалось распарсить JSON:
<SYSTEM INFO>
FinanceAgent специализируется на финансовом анализе, включая анализ выручки, бюджетирование, прогнозирование и финансовую отчетность. ResearchAgent извлекает ключевые факты, сводит источники и предоставляет структурированные знания по любой теме. PlanningAgent раз